Entry: EMA(7) > EMA(30) & Cắt lên MAX(30) & ATR(7) > SMA(ATR, 30)

Stop Loss = Giá Entry - (ATR * 2)

Take Profit = Giá Entry + (ATR * 4)


In [ ]:
!pip install pandas pandas-ta requests -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime

# ==========================================
# 1. GLOBAL CONFIGURATION (Thêm ATR Multipliers)
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'ADAUSDT'],
    'INTERVALS': ['4h', '1d'],
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2025',

    # ATR Exit Settings
    'SL_ATR_MULT': 2.0,                 # Stop Loss = 2 x ATR
    'TP_ATR_MULT': 4.0,                 # Take Profit = 4 x ATR (Tỷ lệ R:R = 1:2)

    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# 2. DATA FETCHER MODULE
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        return int(datetime.strptime(date_str, "%d/%m/%Y").timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        try: requests.get(f"{base_url}/api/v3/time", timeout=5)
        except: base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)
        end_time = int(datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59).timestamp() * 1000)

        all_klines = []
        while start_time < end_time:
            params = {"symbol": self.symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        return df.astype(float)

# ==========================================
# 3. STRATEGY ENGINE (ATR Based Exits)
# ==========================================
class ExplosiveBreakoutATREngine:
    def __init__(self, df, config):
        self.df = df
        self.config = config

    def calculate_indicators(self):
        # 1. Trend Filter
        self.df['EMA_7'] = ta.ema(self.df['Close'], length=7)
        self.df['EMA_30'] = ta.ema(self.df['Close'], length=30)

        # 2. Breakout Levels (Shift 1)
        self.df['MAX_30'] = self.df['High'].rolling(30).max().shift(1)

        # 3. Volatility Filter & ATR Calculator
        self.df['ATR_7'] = ta.atr(self.df['High'], self.df['Low'], self.df['Close'], length=7)
        self.df['SMA_ATR_30'] = ta.sma(self.df['ATR_7'], length=30)

        self.df.dropna(inplace=True)

    def run(self):
        self.calculate_indicators()

        O, H, L, C = self.df['Open'].values, self.df['High'].values, self.df['Low'].values, self.df['Close'].values
        ema7, ema30 = self.df['EMA_7'].values, self.df['EMA_30'].values
        max30 = self.df['MAX_30'].values
        atr7, sma_atr30 = self.df['ATR_7'].values, self.df['SMA_ATR_30'].values
        times = self.df.index

        capital = self.config['INITIAL_CAPITAL']
        fee, slip, pos_size_pct = self.config['FEE_RATE'], self.config['SLIPPAGE'], self.config['POSITION_SIZE_PCT']
        sl_mult, tp_mult = self.config['SL_ATR_MULT'], self.config['TP_ATR_MULT']

        position = 0
        entry_price, sl_price, tp_price = 0, 0, 0
        invested_amt = 0
        trades = []

        for i in range(1, len(self.df)):
            # -------------------------------------
            # EXIT LOGIC (Hit ATR Stop Loss or ATR Take Profit)
            # -------------------------------------
            if position == 1:
                exit_price = 0
                reason = ""

                # Ưu tiên kiểm tra Stoploss trước (Pessimistic execution - an toàn)
                if L[i] <= sl_price:
                    exit_price = sl_price
                    reason = 'Hit_SL'
                elif H[i] >= tp_price:
                    exit_price = tp_price
                    reason = 'Hit_TP'

                if exit_price != 0:
                    real_exit = exit_price * (1 - slip)
                    pnl_pct = (real_exit - entry_price) / entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = invested_amt * net_pnl_pct
                    capital += pnl_amt

                    trades.append({
                        'Date': times[i], 'Type': 'Long', 'Reason': reason,
                        'Entry': entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Capital': capital
                    })
                    position = 0
                    invested_amt = 0
                    continue

            # -------------------------------------
            # ENTRY LOGIC & SET SL/TP
            # -------------------------------------
            if position == 0:
                cond1_trend = ema7[i-1] > ema30[i-1]
                cond2_break = C[i-1] > max30[i-1]
                cond3_vol   = atr7[i-1] > sma_atr30[i-1]

                if cond1_trend and cond2_break and cond3_vol:
                    position = 1
                    entry_price = O[i] * (1 + slip)
                    invested_amt = capital * pos_size_pct

                    # TÍNH TOÁN SL/TP DỰA TRÊN ATR CỦA NẾN BREAKOUT (i-1)
                    current_atr = atr7[i-1]
                    sl_price = entry_price - (current_atr * sl_mult)
                    tp_price = entry_price + (current_atr * tp_mult)

                    # Kiểm tra Intra-bar (Vừa mua xong giá chạy chạm SL/TP luôn trong phiên)
                    exit_price = 0
                    if L[i] <= sl_price: exit_price = sl_price
                    elif H[i] >= tp_price: exit_price = tp_price

                    if exit_price != 0:
                        real_exit = exit_price * (1 - slip)
                        pnl_pct = (real_exit - entry_price) / entry_price
                        pnl_amt = invested_amt * (pnl_pct - fee * 2)
                        capital += pnl_amt

                        trades.append({
                            'Date': times[i], 'Type': 'Long', 'Reason': 'Instant_Exit',
                            'Entry': entry_price, 'Exit': real_exit,
                            'PnL_Amt': pnl_amt, 'Capital': capital
                        })
                        position = 0
                        invested_amt = 0

        return pd.DataFrame(trades)

# ==========================================
# 4. PERFORMANCE EVALUATOR
# ==========================================
def evaluate_performance(trades_df, symbol, interval, config):
    if trades_df.empty: return None

    initial_cap = config['INITIAL_CAPITAL']
    final_cap = trades_df.iloc[-1]['Capital']
    net_pnl = final_cap - initial_cap
    total_return = net_pnl / initial_cap

    wins = trades_df[trades_df['PnL_Amt'] > 0]
    losses = trades_df[trades_df['PnL_Amt'] <= 0]

    trades_count = len(trades_df)
    winrate = len(wins) / trades_count if trades_count > 0 else 0

    gross_profit = wins['PnL_Amt'].sum()
    gross_loss = abs(losses['PnL_Amt'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else (99.99 if gross_profit > 0 else 0)

    trades_df['Peak'] = trades_df['Capital'].cummax()
    trades_df['DD'] = (trades_df['Capital'] - trades_df['Peak']) / trades_df['Peak']
    max_dd = trades_df['DD'].min()

    trades_df.set_index('Date', inplace=True)
    daily_pnl = trades_df.resample('D')['PnL_Amt'].sum()
    daily_capital = initial_cap + daily_pnl.cumsum()
    daily_returns = daily_capital.pct_change().dropna()

    sharpe = 0
    if daily_returns.std() != 0:
        sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365)

    return {
        'Symbol': symbol,
        'Timeframe': interval,
        'Trades': trades_count,
        'Winrate': f"{winrate*100:.2f}%",
        'Profit_Factor': f"{profit_factor:.2f}",
        'Max_DD': f"{max_dd*100:.2f}%",
        'Net_PnL_$': f"{net_pnl:.2f}",
        'Return': f"{total_return*100:.2f}%",
        'Sharpe': f"{sharpe:.2f}"
    }

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print("🚀 Bắt đầu quá trình Backtest Chiến lược: Explosive Breakout")
    print("Logic: Fixed SL = 2 ATR | Fixed TP = 4 ATR (R:R 1:2)\n")

    all_results = []

    for symbol in CONFIG['SYMBOLS']:
        for interval in CONFIG['INTERVALS']:
            fetcher = BinanceDataFetcher(symbol, interval, CONFIG['START_DATE'], CONFIG['END_DATE'])
            df_raw = fetcher.fetch()

            if not df_raw.empty:
                engine = ExplosiveBreakoutATREngine(df_raw, CONFIG)
                trades_df = engine.run()
                metrics = evaluate_performance(trades_df, symbol, interval, CONFIG)
                if metrics:
                    all_results.append(metrics)

    if all_results:
        final_df = pd.DataFrame(all_results)
        final_df['Sharpe_Num'] = final_df['Sharpe'].astype(float)
        final_df = final_df.sort_values('Sharpe_Num', ascending=False).drop('Sharpe_Num', axis=1)

        print("="*90)
        print(" BẢNG PERFORMANCE XẾP HẠNG (RANKED BY SHARPE RATIO)")
        print("="*90)
        print(final_df.to_string(index=False, justify='right'))
        print("="*90)
    else:
        print("Không có tín hiệu giao dịch nào thỏa mãn điều kiện chiến lược.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
🚀 Bắt đầu quá trình Backtest Chiến lược: Explosive Breakout
Logic: Fixed SL = 2 ATR | Fixed TP = 4 ATR (R:R 1:2)

 BẢNG PERFORMANCE XẾP HẠNG (RANKED BY SHARPE RATIO)
 Symbol Timeframe  Trades Winrate Profit_Factor Max_DD Net_PnL_$ Return Sharpe
BTCUSDT        4h      60  55.00%          2.38 -1.41%   1273.81 12.74%   2.10
SOLUSDT        4h      62  48.39%          1.85 -